# 02 — From edge records to chemical–disease pairs

Three questions: **How many pairs exist? Which pairs does our category filter
exclude? What evidence describes one actual pair?**

We use the same eight treatment-related predicates as notebook 01, including
clinical trials and contraindication. Counts describe stored evidence, not confirmed
treatments or model predictions. The original graph and complete selected edge
records remain unchanged.


## How the data is prepared

The next cell runs a [preparation script](../scripts/inspect_treatment_pairs.py).
It reads the compressed graph and keeps an edge only if its predicate exactly
matches one of these eight:

- `biolink:treats`
- `biolink:treats_or_applied_or_studied_to_treat`
- `biolink:applied_to_treat`
- `biolink:studied_to_treat`
- `biolink:in_clinical_trials_for`
- `biolink:in_preclinical_trials_for`
- `biolink:ameliorates_condition`
- `biolink:contraindicated_in`

This includes treatment, research, and contraindication records—not only confirmed treatments.

For those selected edges, the script:

1. Looks up the names and categories of the subject and object nodes.
2. Groups edges with the same ordered `(subject ID, object ID)` into one pair, keeping the original evidence.
3. Selects pairs where the subject has **ChemicalEntity, SmallMolecule, or Drug** in its category list, and the object has **Disease**. One accepted subject label is enough.

Results are saved in `output/treatment_pairs/`: counts in `summary.json`, filtered
pairs in `pairs.csv`, and evidence in `evidence.sqlite`.

Later runs reuse the saved analysis when it is up to date. The original graph is
unchanged; this step does not train a model or create training/test splits.


In [ ]:
from pathlib import Path
import json
from collections import Counter
import sqlite3
import subprocess
import sys

ROOT = next(p / "experiments" / "ultra" for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "experiments" / "ultra" / "download.json").exists())
OUTPUT = ROOT / "output" / "treatment_pairs"
# Cached after the first run. A full scan takes several minutes and writes disk-backed outputs.
subprocess.run([sys.executable, str(ROOT / "scripts" / "inspect_treatment_pairs.py")], check=True)
summary = json.loads((OUTPUT / "summary.json").read_text())
db = sqlite3.connect((OUTPUT / "evidence.sqlite").as_uri() + "?mode=ro", uri=True)
db.row_factory = sqlite3.Row

def show(rows, columns):
    rows = list(rows)
    values = [[str(row[key]) for key in columns] for row in rows]
    widths = [min(65, max([len(key)] + [len(row[i]) for row in values])) for i, key in enumerate(columns)]
    def format_row(row):
        return " | ".join((s if len(s) <= widths[i] else s[:widths[i]-1] + "…").ljust(widths[i])
                          for i, s in enumerate(row))
    print(format_row(columns))
    print("-+-".join("-" * width for width in widths))
    for row in values:
        print(format_row(row))


print(f"Release: {summary['release']}")
print(f"Raw records scanned: {summary['nodes_scanned']:,} nodes; {summary['edges_scanned']:,} edges")
print(f"Counts match publisher totals: {summary['matches_metadata_totals']}")
print(f"Missing endpoint IDs among selected edges: {summary['missing_endpoint_ids']:,}")
assert summary["matches_metadata_totals"]
assert summary["missing_endpoint_ids"] == 0, "Resolve missing endpoints before analyzing exclusions."


Using cached analysis: /Users/carolinagonzalez/shepherd/experiments/ultra/output/treatment_pairs
Release: 2026_06_21
Raw records scanned: 1,754,754 nodes; 28,709,074 edges
Counts match publisher totals: True
Missing endpoint IDs among selected edges: 0


## 1. Edge records versus unique pairs

An **edge record** describes a relationship, including its source and any qualifiers.
A **pair** is the exact ordered `(subject ID, object ID)`. Two records with the same
IDs and predicate but different sources count as **two edges and one pair**.
We do not merge synonyms or reverse directions.

Each row below counts one exact predicate:

- **Edge records:** all records with that predicate, with any endpoint categories.
- **Unique pairs:** distinct subject–object pairs with that predicate, with any categories.
- **Chemical/drug → disease pairs:** those unique pairs passing both conditions below.

**Our category filter:** the subject carries **ChemicalEntity OR SmallMolecule OR Drug**,
and the object carries **Disease**. One accepted subject label is enough. We use raw
category lists, which can include ancestor labels; notebook 01's metadata shows shorter
combinations. The filter does not check approval status, species, or evidence quality.


In [2]:
rows = [{"Predicate": r["predicate"].removeprefix("biolink:"),
         "Edge records": f"{r['edges_all_endpoints']:,}",
         "Unique pairs": f"{r['pairs_all_endpoints']:,}",
         "Chemical/drug → disease pairs": f"{r['eligible_pairs']:,}"}
        for r in summary["predicate_stats"]]
show(rows, ["Predicate", "Edge records", "Unique pairs", "Chemical/drug → disease pairs"])
print(f"\nAcross all eight predicates: {summary['selected_edges']:,} edge records → "
      f"{summary['unique_pairs_all_endpoints']:,} unique pairs → "
      f"{summary['eligible_pairs']:,} pairs passing the category filter.")
print(f"The filtered pairs involve {summary['eligible_chemicals']:,} distinct chemical/drug IDs "
      f"and {summary['eligible_diseases']:,} distinct disease IDs.")
print("Pair-count columns cannot be summed: one pair can occur under multiple predicates.")


Predicate                             | Edge records | Unique pairs | Chemical/drug → disease pairs
--------------------------------------+--------------+--------------+------------------------------
treats_or_applied_or_studied_to_treat | 445,762      | 420,637      | 292,287                      
in_clinical_trials_for                | 429,798      | 428,644      | 310,701                      
applied_to_treat                      | 65,254       | 65,254       | 55,583                       
treats                                | 33,634       | 28,983       | 22,918                       
contraindicated_in                    | 22,859       | 22,125       | 19,705                       
studied_to_treat                      | 1,645        | 1,645        | 1,308                        
in_preclinical_trials_for             | 275          | 275          | 245                          
ameliorates_condition                 | 7            | 7            | 0                            


## 2. Which pairs does the category filter exclude?

Count each excluded pair once across all eight predicates. There are three mutually
exclusive reasons: the subject fails, the object fails, or both fail.
An object labeled PhenotypicFeature without Disease fails even if its name sounds
like a disease. A protein without any accepted chemical/drug label fails as a subject.

The second table groups excluded pairs by readable endpoint labels. These display
groups combine raw category combinations; they do not change the eligibility rule.
Exclusion here does not mean the evidence is wrong or that the node should be removed
from a future ULTRA input graph.


In [3]:
reasons = Counter()
groups = Counter()

def endpoint_group(categories, passes, subject):
    labels = set(json.loads(categories))
    if passes:
        return "Chemical/drug" if subject else "Disease"
    if subject and labels & {"biolink:Gene", "biolink:Protein"}:
        return "Gene/protein (no accepted chemical label)"
    for label in ("PhenotypicFeature", "Procedure", "Device", "Activity", "Cohort",
                  "PopulationOfIndividualOrganisms", "DiseaseOrPhenotypicFeature"):
        if "biolink:" + label in labels:
            return label
    return "Other categories"

for row in db.execute("""
    SELECT s.categories AS subject_categories, o.categories AS object_categories,
           s.is_chemical, o.is_disease, COUNT(*) AS pairs
    FROM pairs p JOIN nodes s ON s.id=p.subject JOIN nodes o ON o.id=p.object
    WHERE s.is_chemical=0 OR o.is_disease=0
    GROUP BY s.categories, o.categories, s.is_chemical, o.is_disease
"""):
    reason = ("Subject only fails" if not row["is_chemical"] and row["is_disease"]
              else "Object only fails" if row["is_chemical"] else "Both fail")
    reasons[reason] += row["pairs"]
    groups[(endpoint_group(row["subject_categories"], row["is_chemical"], True),
            endpoint_group(row["object_categories"], row["is_disease"], False))] += row["pairs"]
excluded = sum(reasons.values())
assert excluded == summary["unique_pairs_all_endpoints"] - summary["eligible_pairs"]
show([{"Reason": reason, "Unique pairs": f"{count:,}"} for reason, count in reasons.most_common()],
     ["Reason", "Unique pairs"])
print(f"\nExcluded: {excluded:,} pairs ({excluded / summary['unique_pairs_all_endpoints']:.1%}).")
print("\nLargest excluded endpoint groups:")
show([{"Subject group": s, "Object group": o, "Unique pairs": f"{count:,}"}
      for (s, o), count in groups.most_common(6)],
     ["Subject group", "Object group", "Unique pairs"])


Reason             | Unique pairs
-------------------+-------------
Subject only fails | 133,007     
Object only fails  | 96,399      
Both fail          | 21,994      

Excluded: 251,400 pairs (28.8%).

Largest excluded endpoint groups:
Subject group                             | Object group      | Unique pairs
------------------------------------------+-------------------+-------------
Chemical/drug                             | PhenotypicFeature | 93,446      
Gene/protein (no accepted chemical label) | Disease           | 88,897      
Procedure                                 | Disease           | 39,531      
Gene/protein (no accepted chemical label) | PhenotypicFeature | 9,538       
Procedure                                 | PhenotypicFeature | 7,205       
Device                                    | Disease           | 2,812       


## 3. One pair, several evidence records

The default example is **Nalidixic acid → bacterial urinary tract infection**.
You can change `PAIR` to another pair in `output/treatment_pairs/pairs.csv`.

Each displayed row below is a separate stored edge. The same pair can have multiple
sources and predicates. Sources are not necessarily independent. These are graph
assertions, not clinical recommendations.


In [4]:
PAIR = ("CHEBI:100147", "MONDO:0005247")
with sqlite3.connect((OUTPUT / "evidence.sqlite").as_uri() + "?mode=ro", uri=True) as evidence_db:
    evidence_db.row_factory = sqlite3.Row
    for role, identifier in zip(("Subject", "Object"), PAIR):
        node = evidence_db.execute("SELECT * FROM nodes WHERE id=?", (identifier,)).fetchone()
        if node is None:
            raise ValueError(f"No indexed node: {identifier}")
        accepted = sorted(set(json.loads(node["categories"])) & {
            "biolink:ChemicalEntity", "biolink:SmallMolecule", "biolink:Drug", "biolink:Disease"})
        print(f"{role}: {node['name']} ({identifier})")
        print("  Matching filter labels: " + (", ".join(x.removeprefix("biolink:") for x in accepted) or "none"))
    records = evidence_db.execute("SELECT raw_json FROM edges WHERE subject=? AND object=? ORDER BY row_id", PAIR)
    examples = []
    for number, record in enumerate(records, 1):
        edge = json.loads(record["raw_json"])
        sources = [s["resource_id"] for s in edge.get("sources", [])
                   if s.get("resource_role") == "primary_knowledge_source" and s.get("resource_id")]
        qualifiers = {k: v for k, v in edge.items() if "qualifier" in k or k == "qualified_predicate"}
        examples.append({"Record": number, "Predicate": edge["predicate"].removeprefix("biolink:"),
                         "Primary sources": ", ".join(sources) or "not recorded",
                         "Qualifiers": json.dumps(qualifiers) if qualifiers else "none recorded"})
    print()
    show(examples[:10], ["Record", "Predicate", "Primary sources", "Qualifiers"])
    if examples:
        print(f"\n{len(examples)} selected edge records connect this one ordered pair; showing up to 10.")
    else:
        print("No selected edge records for this pair.")
    print("Full edge records, including publications and source details, remain in evidence.sqlite.")


Subject: Nalidixic acid (CHEBI:100147)
  Matching filter labels: ChemicalEntity, Drug, SmallMolecule
Object: bacterial urinary tract infection (MONDO:0005247)
  Matching filter labels: Disease

Record | Predicate                             | Primary sources                       | Qualifiers   
-------+---------------------------------------+---------------------------------------+--------------
1      | treats                                | infores:drug-repurposing-hub          | none recorded
2      | treats                                | infores:drugcentral                   | none recorded
3      | treats_or_applied_or_studied_to_treat | infores:text-mining-provider-targeted | none recorded

3 selected edge records connect this one ordered pair; showing up to 10.
Full edge records, including publications and source details, remain in evidence.sqlite.


The category filter is our working choice, not a final definition of the prediction
task. Before building ULTRA inputs, decide whether to include phenotype targets or
other therapeutic entity types, then examine their biological connections.

The cached analysis is in `output/treatment_pairs/`: `summary.json`, `pairs.csv`,
and `evidence.sqlite`. Detailed overlap metrics remain available there for later analysis.
